In [1]:
import imaplib
import email
from email.header import decode_header
import os



In [7]:
# Replace with your actual Gmail address and password
EMAIL_ADDRESS = "suryaansh202@gmail.com"
EMAIL_PASSWORD = "bofm aesi dcvl xpzh"

# Directory to save attachments
ATTACHMENT_DIR = "attachments"
if not os.path.exists(ATTACHMENT_DIR):
    os.makedirs(ATTACHMENT_DIR)


In [10]:
def clean(text):
    """Clean text by removing newline and carriage return characters."""
    return text.replace('\n', ' ').replace('\r', ' ')


In [11]:
def decode_header_field(field):
    """Decode a header field into a readable string."""
    try:
        decoded_parts = decode_header(field)
        decoded_string = ""
        for part, encoding in decoded_parts:
            if isinstance(part, bytes):
                decoded_string += part.decode(encoding or "utf-8", errors="ignore")
            else:
                decoded_string += part
        return decoded_string
    except Exception:
        return field


In [12]:
def read_emails():
    """
    Connects to Gmail via IMAP, reads emails from the inbox, and returns a list of structured email data.
    
    Each email is represented as a dictionary with keys:
      - subject: The email's subject.
      - from: The sender.
      - date: The date string.
      - body: The decoded text/plain (or HTML fallback) body.
      - attachments: A list of attachments, where each attachment is a dictionary
                     with 'filename' and 'content' keys.
    """
    emails = []
    
    try:
        with imaplib.IMAP4_SSL("imap.gmail.com") as mail:
            mail.login(EMAIL_ADDRESS, EMAIL_PASSWORD)
            mail.select("inbox")
            
            status, email_ids = mail.search(None, "ALL")
            if status != "OK":
                print("Error searching emails:", status)
                return emails
            
            for email_id in email_ids[0].split():
                status, msg_data = mail.fetch(email_id, "(RFC822)")
                if status != "OK":
                    continue
                
                # Parse the raw email content
                msg = email.message_from_bytes(msg_data[0][1])
                email_dict = {}
                
                # Decode subject, sender, and date headers
                subject = msg.get("Subject", "No Subject")
                email_dict["subject"] = clean(decode_header_field(subject))
                
                from_ = msg.get("From", "Unknown Sender")
                email_dict["from"] = clean(decode_header_field(from_))
                
                email_dict["date"] = msg.get("Date")
                
                body = ""
                attachments = []
                
                # If the message is multipart, iterate through its parts
                if msg.is_multipart():
                    for part in msg.walk():
                        content_type = part.get_content_type()
                        disposition = part.get("Content-Disposition") or ""
                        
                        # Check if this part is an attachment
                        if "attachment" in disposition.lower():
                            filename = part.get_filename()
                            if filename:
                                filename = decode_header_field(filename)
                            else:
                                filename = "attachment"
                            
                            payload = part.get_payload(decode=True)
                            # Try to decode as text; if it fails, keep binary data
                            try:
                                att_content = payload.decode("utf-8", errors="ignore")
                            except Exception:
                                att_content = payload
                            
                            attachments.append({"filename": filename, "content": att_content})
                        
                        # For the body, try text/plain first
                        elif content_type == "text/plain" and not body:
                            try:
                                body = part.get_payload(decode=True).decode("utf-8", errors="ignore")
                            except Exception:
                                body = ""
                        # Optionally, fallback to text/html if no plain text found
                        elif content_type == "text/html" and not body:
                            try:
                                body = part.get_payload(decode=True).decode("utf-8", errors="ignore")
                            except Exception:
                                body = ""
                else:
                    # For non-multipart messages, handle accordingly
                    content_type = msg.get_content_type()
                    if content_type == "text/plain":
                        try:
                            body = msg.get_payload(decode=True).decode("utf-8", errors="ignore")
                        except Exception:
                            body = ""
                    else:
                        # If it's not plain text, treat it as an attachment
                        filename = msg.get_filename()
                        if filename:
                            filename = decode_header_field(filename)
                        else:
                            filename = "attachment"
                        
                        payload = msg.get_payload(decode=True)
                        try:
                            att_content = payload.decode("utf-8", errors="ignore")
                        except Exception:
                            att_content = payload
                        attachments.append({"filename": filename, "content": att_content})
                
                email_dict["body"] = clean(body)
                email_dict["attachments"] = attachments
                
                emails.append(email_dict)
                
        return emails
        
    except Exception as e:
        print("An error occurred:", e)
        return emails


In [29]:
emails = read_emails()


In [30]:
len(emails)

6

In [31]:
emails[-1]

{'subject': 'Data File',
 'from': 'Suryaansh Rathinam <suryaansh28@gmail.com>',
 'date': 'Fri, 4 Apr 2025 13:50:37 +0800',
 'body': '  ',
 'attachments': [{'filename': 'dummy_data.csv',
   'content': 'ID,Name,Email,Phone,Address,DateOfBirth,JoinDate,Salary\na354fd3a-999b-451a-8034-ecd6c25e95c2,Beth Malone,danamills@brown.net,001-132-364-6063x250,"94812 Maria Crest, Paulport, CA 91632",1987-01-01,2023-03-16,67823.12\n2eee25b3-2a2a-461f-be4a-0411dab0f5c4,Lynn White,dawn54@yahoo.com,001-898-695-5837,"23001 Anderson Streets Apt. 812, Georgeside, SC 61586",1973-08-02,2022-10-01,69078.86\nd75b8ef8-62e3-4dec-9d9c-e766d6cb5589,Mary Henry,dixonpamela@yahoo.com,524.870.6096,"3197 Welch Springs, Lake Ronaldberg, IL 96435",1971-03-28,2025-02-02,44937.89\n3c4ced4a-f6d2-4d50-9d8e-1324888b2435,Brianna Turner,wardlisa@young.com,676-669-3266x76556,"Unit 8643 Box 3618, DPO AE 79113",2005-07-08,2023-04-13,72020.9\n7641e391-fc7c-4843-83fd-ab091098d89d,April Hamilton,csullivan@strickland.net,001-801-051-57